In [3]:
!nvidia-smi
!python --version
!ls /kaggle/input
!ls /kaggle/input/*

Wed Sep  9 11:54:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!ls -R /kaggle/input | head -30

/kaggle/input:
datasets

/kaggle/input/datasets:
bornamuzina

/kaggle/input/datasets/bornamuzina:
dataset

/kaggle/input/datasets/bornamuzina/dataset:
config.py
dataset.py
evaluate.py
targets.py
tensors_rgb_packed_k
train.py

/kaggle/input/datasets/bornamuzina/dataset/tensors_rgb_packed_k:
tensors_rgb_packed

/kaggle/input/datasets/bornamuzina/dataset/tensors_rgb_packed_k/tensors_rgb_packed:
splits.json
V_DRONE_001_meta.json
V_DRONE_001_tensors.npz
V_DRONE_002_meta.json
V_DRONE_002_tensors.npz
V_DRONE_003_meta.json
V_DRONE_003_tensors.npz
V_DRONE_004_meta.json
V_DRONE_004_tensors.npz


In [6]:
BASE = '/kaggle/input/datasets/bornamuzina/dataset'
DATA = BASE + '/tensors_rgb_packed_k/tensors_rgb_packed'
CODE = BASE

!ls {CODE}/*.py
!ls {DATA} | head
!ls {CODE}/*.py
!ls {DATA} | head

/kaggle/input/datasets/bornamuzina/dataset/config.py
/kaggle/input/datasets/bornamuzina/dataset/dataset.py
/kaggle/input/datasets/bornamuzina/dataset/evaluate.py
/kaggle/input/datasets/bornamuzina/dataset/targets.py
/kaggle/input/datasets/bornamuzina/dataset/train.py
splits.json
V_DRONE_001_meta.json
V_DRONE_001_tensors.npz
V_DRONE_002_meta.json
V_DRONE_002_tensors.npz
V_DRONE_003_meta.json
V_DRONE_003_tensors.npz
V_DRONE_004_meta.json
V_DRONE_004_tensors.npz
V_DRONE_005_meta.json
/kaggle/input/datasets/bornamuzina/dataset/config.py
/kaggle/input/datasets/bornamuzina/dataset/dataset.py
/kaggle/input/datasets/bornamuzina/dataset/evaluate.py
/kaggle/input/datasets/bornamuzina/dataset/targets.py
/kaggle/input/datasets/bornamuzina/dataset/train.py
splits.json
V_DRONE_001_meta.json
V_DRONE_001_tensors.npz
V_DRONE_002_meta.json
V_DRONE_002_tensors.npz
V_DRONE_003_meta.json
V_DRONE_003_tensors.npz
V_DRONE_004_meta.json
V_DRONE_004_tensors.npz
V_DRONE_005_meta.json


In [8]:
!cp {CODE}/*.py /kaggle/working/
%cd /kaggle/working
!ls *.py

/kaggle/working
config.py  dataset.py  evaluate.py  targets.py	train.py


In [9]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /kaggle/working/akv
!/kaggle/working/akv/bin/pip install -q --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 11.6 MB/s eta 0:00:0000:0100:01


In [10]:
!/kaggle/working/akv/bin/pip install -q akida-models==1.14.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [11]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -c "import tensorflow as tf; print(tf.__version__); print(tf.config.list_physical_devices('GPU'))"

2026-09-09 12:00:47.566020: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788955247.587515     330 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788955247.594218     330 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788955247.610818     330 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788955247.610865     330 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788955247.610872     330 computation_placer.cc:177] computation placer alr

In [12]:
import re

src = open('config.py').read()
src = re.sub(r"ROOT = Path\(.*?\)", "ROOT = Path('/kaggle/working')", src)
src = re.sub(r"TENSOR_DIR = .*", f"TENSOR_DIR = Path('{DATA}')", src)
src = re.sub(r"SPLITS_FILE = .*", f"SPLITS_FILE = Path('{DATA}/splits.json')", src)
src = re.sub(r"RUNS_DIR = .*", "RUNS_DIR = Path('/kaggle/working/runs')", src)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

print(open('config.py').read()[:600])

"""
Every constant that has to agree across the pipeline.

Anchors in particular must be identical in target creation and in
inference decoding. If they drift apart the network trains fine and
then predicts boxes of the wrong size, which looks like a broken model
rather than a broken constant. Keeping them in one file that both sides
import removes that failure mode.
"""

from pathlib import Path


# ============================================================
# PATHS
# ============================================================

ROOT = Path('/kaggle/working')
TENSOR_DIR = Path('/kaggle/input


In [13]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python dataset.py

tensors : /kaggle/input/datasets/bornamuzina/dataset/tensors_rgb_packed_k/tensors_rgb_packed
policy  : keep

TRAIN  (90 clips in split)
  clips loaded : 90
  samples      : 28045
  quiet frames : 0 (policy: keep)
  boxes        : 28345
  box size     : median 10.4 px (0.32 cells), p5 6.7, p95 17.7
  under 8 px   : 16.4%

VALIDATION  (12 clips in split)
  clips loaded : 12
  samples      : 3728
  quiet frames : 0 (policy: keep)
  boxes        : 3728
  box size     : median 10.2 px (0.32 cells), p5 6.5, p95 18.2
  under 8 px   : 25.0%

TEST  (12 clips in split)
  clips loaded : 12
  samples      : 3723
  quiet frames : 0 (policy: keep)
  boxes        : 4243
  box size     : median 10.6 px (0.33 cells), p5 8.1, p95 20.5
  under 8 px   : 4.6%

One batch:
  images shape : (4, 224, 224, 3)  float32
  value range  : 0.00 to 227.00
  zero pixels  : 20.1%
  boxes/frame  : [1, 1, 1, 1]
  out of bounds: 0
  inside padding: 0


In [15]:
!grep -n "add_argument" train.py

230:    ap.add_argument("--clips", type=int, default=0,
232:    ap.add_argument("--epochs", type=int, default=config.EPOCHS)
233:    ap.add_argument("--batch_size", type=int, default=config.BATCH_SIZE)
234:    ap.add_argument("--lr", type=float, default=config.LEARNING_RATE)
235:    ap.add_argument("--alpha", type=float, default=0.5)
236:    ap.add_argument("--tensors", default=None,
238:    ap.add_argument("--name", default="run")


In [16]:
!grep -n "yolo_base\|akidanet\|alpha" train.py

6:1. Inputs are mapped to uint8 before the model sees them. yolo_base has
235:    ap.add_argument("--alpha", type=float, default=0.5)
241:    from akida_models import yolo_base
261:    model = yolo_base(
265:        alpha=args.alpha,
269:    print(f"model    : alpha {args.alpha}, {model.count_params():,} params")


In [18]:
!sed -n '255,275p' train.py

    val_ds = dataset.EventDataset(splits["validation"], tensor_dir=tensor_dir)

    if len(train_ds) == 0:
        raise SystemExit("No training samples.")

    # ---- model -------------------------------------------------------
    model = yolo_base(
        input_shape=(config.INPUT_SIZE, config.INPUT_SIZE, config.CHANNELS),
        classes=config.N_CLASSES,
        nb_box=config.N_ANCHORS,
        alpha=args.alpha,
    )

    print()
    print(f"model    : alpha {args.alpha}, {model.count_params():,} params")
    print(f"output   : {model.output_shape}")
    print(f"clip     : +/-{CLIP} -> uint8")
    print(f"lr       : {args.lr}, batch {args.batch_size}")
    print()

    optimizer = tf.keras.optimizers.Adam(learning_rate=args.lr)


In [21]:
src = open('train.py').read()
src = src.replace(
    "    x = np.clip(images, -clip, clip)\n    return (x / clip * 127.0 + 128.0).astype(np.float32)",
    "    if images.dtype == np.uint8:\n        return images.astype(np.float32)\n    x = np.clip(images, -clip, clip)\n    return (x / clip * 127.0 + 128.0).astype(np.float32)"
)
open('train.py','w').write(src)
!sed -n '/def to_uint8/,/astype/p' train.py

def to_uint8(images, clip=CLIP):
    """
    Signed accumulation -> 0-255.

    Zero maps to 128, so 'nothing happened' sits at the midpoint and the
    two polarities occupy the halves either side of it.
    """
    if images.dtype == np.uint8:
        return images.astype(np.float32)


In [23]:
!grep -n "def to_uint8" -A 10 evaluate.py

In [24]:
!grep -n "uint8\|clip\|CLIP\|import" evaluate.py | head -20

24:from pathlib import Path
25:import argparse
26:import json
28:import numpy as np
29:import tensorflow as tf
31:import config
32:import dataset
33:import train as trainlib
73:                w = np.exp(np.clip(p[2], -8, 8)) * anchors[a, 0] * cell
74:                h = np.exp(np.clip(p[3], -8, 8)) * anchors[a, 1] * cell
226:    from akida_models import yolo_base
280:        x = trainlib.to_uint8(np.stack(images))


In [27]:
!/kaggle/working/akv/bin/python -c "import numpy as np; d = np.load('/kaggle/input/datasets/bornamuzina/dataset/tensors_rgb_packed_k/tensors_rgb_packed/V_DRONE_001_tensors.npz')['a']; print(d.dtype, d.shape, d.min(), d.max())"

uint8 (301, 224, 224, 3) 0 205


In [32]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -c "import numpy as np, dataset, json, train as t; s=dataset.load_splits(); ds=dataset.EventDataset(s['validation']); item=ds[0]; print('type', type(item), len(item)); x=item[0]; print('raw', x.dtype, x.shape, x.min(), x.max()); y=t.to_uint8(np.stack([x])); print('after', y.dtype, y.min(), y.max())"

2026-09-09 13:08:57.987802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788959338.010805     787 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788959338.018364     787 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788959338.036045     787 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788959338.036097     787 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788959338.036105     787 computation_placer.cc:177] computation placer alr

In [33]:
src = open('train.py').read()
src = src.replace(
    "    if images.dtype == np.uint8:\n        return images.astype(np.float32)",
    "    if images.min() >= 0 and images.max() > 5:\n        return images.astype(np.float32)"
)
open('train.py','w').write(src)
!sed -n '/def to_uint8/,/128.0/p' train.py

def to_uint8(images, clip=CLIP):
    """
    Signed accumulation -> 0-255.

    Zero maps to 128, so 'nothing happened' sits at the midpoint and the
    two polarities occupy the halves either side of it.
    """
    if images.min() >= 0 and images.max() > 5:
        return images.astype(np.float32)
    x = np.clip(images, -clip, clip)
    return (x / clip * 127.0 + 128.0).astype(np.float32)


In [34]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -c "import numpy as np, dataset, json, train as t; s=dataset.load_splits(); ds=dataset.EventDataset(s['validation']); item=ds[0]; print('type', type(item), len(item)); x=item[0]; print('raw', x.dtype, x.shape, x.min(), x.max()); y=t.to_uint8(np.stack([x])); print('after', y.dtype, y.min(), y.max())"

2026-09-09 13:10:10.297515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788959410.319485     803 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788959410.326830     803 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788959410.344786     803 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788959410.344831     803 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788959410.344838     803 computation_placer.cc:177] computation placer alr

In [35]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u train.py --epochs 30 --lr 1e-3 --batch_size 64 --name full_rgb

2026-09-09 13:12:15.905710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788959535.928623     812 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788959535.935768     812 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788959535.953319     812 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788959535.953345     812 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788959535.953349     812 computation_placer.cc:177] computation placer alr

In [20]:
!grep -n "CLIP" train.py
!sed -n '/def to_uint8/,/return/p' train.py

36:CLIP = 5.0
43:def to_uint8(images, clip=CLIP):
271:    print(f"clip     : +/-{CLIP} -> uint8")
321:                "clip": CLIP,
def to_uint8(images, clip=CLIP):
    """
    Signed accumulation -> 0-255.

    Zero maps to 128, so 'nothing happened' sits at the midpoint and the
    two polarities occupy the halves either side of it.
    """
    x = np.clip(images, -clip, clip)
    return (x / clip * 127.0 + 128.0).astype(np.float32)


In [ ]:
import json
h = json.load(open('/kaggle/working/runs/full_rgb/history.json'))
for e in h['history']:
    print(f"{e['epoch']:3d}  train {e['train_loss']:7.3f}  val {e['val_loss']:7.3f}  "
          f"t.rec {e['train_recall']:.3f}  v.rec {e['val_recall']:.3f}  "
          f"maxobj {e['val_max_objectness']:.3f}")

In [36]:
!MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py --run full_rgb --split validation

2026-09-09 14:04:34.072366: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788962674.097217     932 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788962674.105795     932 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788962674.128781     932 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788962674.128807     932 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788962674.128811     932 computation_placer.cc:177] computation placer alr

In [38]:
!cd /kaggle/working && zip -qr full_rgb.zip runs/full_rgb
!ls -lh /kaggle/working/full_rgb.zip

-rw-r--r-- 1 root root 13M Sep  9 14:10 /kaggle/working/full_rgb.zip
